# 04 Transformer Extension - To be implemented

# 04 - Transformer Extension: DistilBERT for Sentiment Analysis

**Objective:** Fine-tune a pre-trained DistilBERT model for binary sentiment classification (positive vs negative) on the GoEmotions dataset.


---

## Overview

This notebook implements the extension model required by the project rubric:
- Load the preprocessed dataset (500 positive + 500 negative comments)
- Tokenize text using DistilBERT tokenizer
- Prepare PyTorch DataLoaders for training
- Visualize tokenization examples and sequence length distributions

In [ ]:
import os
import sys
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'balanced_dataset.csv')
OUTPUTS_PATH = os.path.join(PROJECT_ROOT, 'outputs')

MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 64
BATCH_SIZE = 16
TEST_SIZE = 0.2

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:

print(f"Loading dataset from: {DATA_PATH}")

if not os.path.exists(DATA_PATH):
    print("\n️  Dataset not found. Creating a synthetic sample for development...")
    sample_texts = [
        "This is absolutely wonderful, thank you!",
        "I hate this, it's terrible and disappointing.",
        "What a beautiful and joyful experience.",
        "This makes me so angry and frustrated.",
        "I'm so grateful and full of admiration.",
        "Just watching this upsets me deeply.",
    ] * 100
    sample_labels = [1, 0, 1, 0, 1, 0] * 100

    df = pd.DataFrame({'text': sample_texts[:1000], 'label': sample_labels[:1000]})
    print(f"Created synthetic dataset with {len(df)} samples")
else:
    df = pd.read_csv(DATA_PATH)
    print(f" Loaded dataset with {len(df)} samples")

print(f"\nDataset shape: {df.shape}")
print(f"Class distribution:\n{df['label'].value_counts()}")
print(f"\nFirst 5 samples:")
df.head()

In [ ]:
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Special tokens: {tokenizer.all_special_tokens}")
print(f"Max model length: {tokenizer.model_max_length}")

In [ ]:

example_comments = [
    "A beautiful poem for a beautiful thing. Happy cake day btw",
    "Just watching this happen upsets me",
    "I still don't know what I want to be when I grow up",
    "This is absolutely amazing, I love it!️",
]

print("=" * 80)
print("TOKENIZATION EXAMPLES")
print("=" * 80)

for i, text in enumerate(example_comments, 1):
    print(f"\n--- Example {i} ---")
    print(f"Original text: {text}")

    encoded = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_attention_mask=True,
        return_tensors='pt'
    )

    print(f"Input IDs shape: {encoded['input_ids'].shape}")
    print(f"Input IDs: {encoded['input_ids'][0].tolist()}")
    print(f"Decoded tokens: {tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])}")
    print(f"Attention mask: {encoded['attention_mask'][0].tolist()}")

In [ ]:
print("Analyzing token lengths across the entire dataset...")

lengths = []
for text in df['text']:
    encoded = tokenizer(text, truncation=False)
    lengths.append(len(encoded['input_ids']))

lengths = np.array(lengths)

print(f"\nSequence length statistics:")
print(f"  Min: {lengths.min()}")
print(f"  Max: {lengths.max()}")
print(f"  Mean: {lengths.mean():.2f}")
print(f"  Median: {np.median(lengths):.2f}")
print(f"  95th percentile: {np.percentile(lengths, 95):.2f}")
print(f"  99th percentile: {np.percentile(lengths, 99):.2f}")

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(lengths, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
plt.axvline(MAX_LENGTH, color='red', linestyle='--', linewidth=2, label=f'MAX_LENGTH={MAX_LENGTH}')
plt.axvline(lengths.mean(), color='green', linestyle='--', linewidth=2, label=f'Mean={lengths.mean():.1f}')
plt.xlabel('Number of tokens')
plt.ylabel('Frequency')
plt.title('Distribution of Sequence Lengths')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(lengths, bins=50, cumulative=True, density=True, color='steelblue', edgecolor='black', alpha=0.7)
plt.axvline(MAX_LENGTH, color='red', linestyle='--', linewidth=2, label=f'MAX_LENGTH={MAX_LENGTH}')
plt.xlabel('Number of tokens')
plt.ylabel('Cumulative proportion')
plt.title('Cumulative Distribution of Sequence Lengths')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, 'figures', '04_sequence_length_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

truncated = (lengths > MAX_LENGTH).sum()
print(f"\n  Sequences that would be truncated: {truncated}/{len(lengths)} ({100*truncated/len(lengths):.2f}%)")

In [ ]:
class GoEmotionsDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_length=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

print(" Dataset class defined successfully")

In [ ]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=df['label']
)

print(f"Train set: {len(train_texts)} samples")
print(f"Test set: {len(test_texts)} samples")
print(f"\nTrain class distribution: {np.bincount(train_labels)}")
print(f"Test class distribution: {np.bincount(test_labels)}")

train_dataset = GoEmotionsDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
test_dataset = GoEmotionsDataset(test_texts, test_labels, tokenizer, MAX_LENGTH)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"\n Train DataLoader: {len(train_loader)} batches")
print(f" Test DataLoader: {len(test_loader)} batches")

print("\n--- Sample batch ---")
batch = next(iter(train_loader))
print(f"input_ids shape: {batch['input_ids'].shape}")
print(f"attention_mask shape: {batch['attention_mask'].shape}")
print(f"labels shape: {batch['label'].shape}")
print(f"Sample labels: {batch['label'].tolist()}")

In [ ]:

os.makedirs(os.path.join(OUTPUTS_PATH, 'processed'), exist_ok=True)

split_info = {
    'train_size': len(train_texts),
    'test_size': len(test_texts),
    'test_size_ratio': TEST_SIZE,
    'random_seed': SEED,
    'max_length': MAX_LENGTH,
    'batch_size': BATCH_SIZE,
    'model_name': MODEL_NAME
}

import json
with open(os.path.join(OUTPUTS_PATH, 'processed', '04_tokenization_config.json'), 'w') as f:
    json.dump(split_info, f, indent=2)

print(" Tokenization configuration saved")
print(f"\nConfiguration:")
for key, value in split_info.items():
    print(f"  {key}: {value}")

print("\n" + "=" * 0)

## Transformer Training and Evaluation

In [ ]:
import os
import sys
from pathlib import Path

# Locate the project root regardless of the directory used to start Jupyter.
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').is_dir():
    raise FileNotFoundError('Could not locate the project root containing the src folder.')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from models import TransformerSentiment
from train import train_transformer
from evaluate import evaluate_transformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = TransformerSentiment(
    model_name=MODEL_NAME,
    num_classes=2,
    dropout=0.3
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
EPOCHS = 5
LEARNING_RATE = 2e-5

print(f"Starting training for {EPOCHS} epochs...")
print(f"Learning rate: {LEARNING_RATE}")
print("=" * 60)

model, history = train_transformer(
    model=model,
    train_loader=train_loader,
    val_loader=test_loader,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
    device=device
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Transformer Training Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Transformer Training Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, 'figures', '04_transformer_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("=" * 60)
print("TRANSFORMER EVALUATION ON TEST SET")
print("=" * 60)

metrics, preds, labels = evaluate_transformer(model, test_loader, device)

print(f"Accuracy:  {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall:    {metrics['recall']:.4f}")
print(f"F1-Score:  {metrics['f1']:.4f}")
print(f"Kappa:     {metrics['kappa']:.4f}")
print("\nClassification Report:")
print(metrics['classification_report'])

In [ ]:
import seaborn as sns

cm = metrics['confusion_matrix']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Transformer Confusion Matrix')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_PATH, 'figures', '04_transformer_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import json

os.makedirs(os.path.join(OUTPUTS_PATH, 'metrics'), exist_ok=True)
os.makedirs(os.path.join(OUTPUTS_PATH, 'saved_models'), exist_ok=True)

transformer_metrics = {
    'model': 'DistilBERT',
    'accuracy': metrics['accuracy'],
    'precision': metrics['precision'],
    'recall': metrics['recall'],
    'f1': metrics['f1'],
    'kappa': metrics['kappa'],
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'batch_size': BATCH_SIZE,
    'max_length': MAX_LENGTH
}

with open(os.path.join(OUTPUTS_PATH, 'metrics', 'transformer_metrics.json'), 'w') as f:
    json.dump(transformer_metrics, f, indent=2)

torch.save(model.state_dict(), os.path.join(OUTPUTS_PATH, 'saved_models', 'transformer_model.pt'))

print("Metrics saved to: outputs/metrics/transformer_metrics.json")
print("Model saved to: outputs/saved_models/transformer_model.pt")